# Maghrib News Aggregator - Data Exploration

Ce notebook vous permet d'explorer les données collectées par l'agrégateur.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Configuration
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Charger les données

In [ ]:
# Connexion à la base de données
conn = sqlite3.connect('news.db')

# Charger tous les articles
df = pd.read_sql_query("SELECT * FROM articles", conn)

# Convertir les colonnes de date
df['scraped_at'] = pd.to_datetime(df['scraped_at'])
df['published_at'] = pd.to_datetime(df['published_at'])

print(f"Total d'articles: {len(df)}")
df.head()

## 2. Statistiques générales

In [ ]:
# Articles par source
print("Articles par source:")
print(df['source'].value_counts())
print()

# Articles par catégorie
print("Articles par catégorie:")
print(df['category'].value_counts())
print()

# Articles par sentiment
print("Articles par sentiment:")
print(df['sentiment_label'].value_counts())

## 3. Visualisations

In [ ]:
# Distribution par source
plt.figure(figsize=(10, 6))
df['source'].value_counts().plot(kind='bar', color='steelblue')
plt.title('Nombre d\'articles par source', fontsize=14, fontweight='bold')
plt.xlabel('Source')
plt.ylabel('Nombre d\'articles')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution par catégorie
plt.figure(figsize=(12, 6))
df['category'].value_counts().head(10).plot(kind='barh', color='coral')
plt.title('Top 10 Catégories', fontsize=14, fontweight='bold')
plt.xlabel('Nombre d\'articles')
plt.ylabel('Catégorie')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution du sentiment
plt.figure(figsize=(8, 6))
sentiment_counts = df['sentiment_label'].value_counts()
colors = {'Positif': 'green', 'Neutre': 'gray', 'Négatif': 'red'}
sentiment_colors = [colors.get(label, 'blue') for label in sentiment_counts.index]
sentiment_counts.plot(kind='pie', autopct='%1.1f%%', colors=sentiment_colors)
plt.title('Distribution du Sentiment', fontsize=14, fontweight='bold')
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Articles au fil du temps
plt.figure(figsize=(14, 6))
df.set_index('scraped_at').resample('D').size().plot(kind='line', marker='o')
plt.title('Articles scrapés par jour', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Nombre d\'articles')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Analyse de sentiment par source

In [ ]:
# Heatmap sentiment par source
sentiment_by_source = pd.crosstab(df['source'], df['sentiment_label'])

plt.figure(figsize=(10, 6))
sns.heatmap(sentiment_by_source, annot=True, fmt='d', cmap='YlOrRd')
plt.title('Sentiment par Source', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Top articles positifs et négatifs

In [ ]:
# Top 5 articles positifs
print("=== TOP 5 ARTICLES POSITIFS ===")
positive_articles = df.nlargest(5, 'sentiment_score')[['title', 'source', 'sentiment_score', 'url']]
for idx, row in positive_articles.iterrows():
    print(f"\n{row['source']} - Score: {row['sentiment_score']:.2f}")
    print(f"Titre: {row['title']}")
    print(f"URL: {row['url']}")

In [ ]:
# Top 5 articles négatifs
print("=== TOP 5 ARTICLES NÉGATIFS ===")
negative_articles = df.nsmallest(5, 'sentiment_score')[['title', 'source', 'sentiment_score', 'url']]
for idx, row in negative_articles.iterrows():
    print(f"\n{row['source']} - Score: {row['sentiment_score']:.2f}")
    print(f"Titre: {row['title']}")
    print(f"URL: {row['url']}")

## 6. Recherche et filtrage

In [ ]:
# Rechercher des articles par mot-clé
keyword = "économie"  # Changez ce mot-clé

results = df[df['title'].str.contains(keyword, case=False, na=False) | 
             df['content'].str.contains(keyword, case=False, na=False)]

print(f"Résultats pour '{keyword}': {len(results)} articles")
print("\nTitres:")
for title in results['title'].head(10):
    print(f"- {title}")

In [ ]:
# Fermer la connexion
conn.close()